# Dateien lesen und schreiben

Bisher standen die Daten unserer Programme fest im Code oder wurden per `input()` eingegeben; nach Programmende waren alle Ergebnisse verloren. In der Praxis kommen Messwerte aber oft aus Dateien, die ein Datenlogger oder Prüfstand geschrieben hat, und Auswertungen sollen gespeichert und weitergegeben werden.

In diesem Anhang lernen Sie,

- wie man Textdateien schreibt und liest,
- wie man Pfade mit dem Modul `pathlib` sauber angibt und
- wie man CSV-Dateien mit dem Modul `csv` verarbeitet.

Im Kapitel *Bibliotheken* haben Sie mit `pd.read_csv()` bereits eine sehr bequeme Möglichkeit kennengelernt, Tabellen einzulesen. Hier schauen wir, was darunter passiert.

## Dateien, Pfade und das Arbeitsverzeichnis

Eine Datei wird über ihren **Pfad** angesprochen. **Absolute Pfade** gehen vom Laufwerk bzw. Wurzelverzeichnis aus (z.B. `C:\Users\weber\Messungen\lauf1.txt`), **relative Pfade** vom aktuellen **Arbeitsverzeichnis** (z.B. `data/lauf1.txt`). Bei einem Jupyter Notebook ist das normalerweise der Ordner, in dem das Notebook liegt. Wir verwenden relative Pfade in den Unterordner `data`, in dem auch die Prüfstandsdaten aus dem Kapitel *Bibliotheken* liegen.

Für Pfade empfiehlt sich das Modul `pathlib` aus der Standardbibliothek. Ein `Path`-Objekt wird mit dem Operator `/` zusammengesetzt – unabhängig davon, ob das Betriebssystem `/` oder `\` als Trennzeichen verwendet:

In [1]:
from pathlib import Path

print("Arbeitsverzeichnis:", Path.cwd().name)  # .name: nur der letzte Teil des Pfads

data_dir = Path("data")
protocol_file = data_dir / "messprotokoll.txt"  # Pfade werden mit / zusammengesetzt

print(protocol_file)
print("Dateiname:", protocol_file.name)
print("Endung:", protocol_file.suffix)
print("Ordner existiert?", data_dir.exists())

Arbeitsverzeichnis: notebooks
data/messprotokoll.txt
Dateiname: messprotokoll.txt
Endung: .txt
Ordner existiert? True


::: {.callout-important}
### Windows-Pfade als String {.unnumbered}

Wer Pfade als normale Strings schreibt, stolpert unter Windows über den Backslash: In `"C:\Daten\neu.txt"` wird `\n` als Zeilenumbruch interpretiert! Abhilfe schaffen ein **Raw String** `r"C:\Daten\neu.txt"`, normale Schrägstriche `"C:/Daten/neu.txt"` (funktionieren auch unter Windows) oder eben `Path`.
:::

## Textdateien schreiben

Eine Datei wird mit der eingebauten Funktion `open()` geöffnet. Neben dem Pfad gibt man den **Modus** an:

| Modus | Bedeutung |
|---|---|
| `"r"` | lesen (read) – Standard; Fehler, wenn die Datei nicht existiert |
| `"w"` | schreiben (write) – legt die Datei an, **überschreibt** eine bestehende Datei! |
| `"a"` | anhängen (append) – schreibt ans Ende einer bestehenden Datei |

Die `with`-Anweisung schließt die Datei automatisch, sobald der eingerückte Block verlassen wird – auch dann, wenn darin ein Fehler auftritt. Ohne Schließen landen Daten unter Umständen nicht vollständig auf der Festplatte.

Wir schreiben ein kleines Messprotokoll eines Temperatursensors:

In [2]:
times_min = [0, 5, 10, 15, 20, 25, 30]
temperatures = [21.4, 23.9, 27.5, 31.2, 34.8, 36.1, 35.7]

with open(protocol_file, "w", encoding="utf-8") as f:
    f.write("# Messprotokoll Temperatursensor T1\n")  # \n: Zeilenumbruch selbst einfügen!
    f.write("# Zeit [min], Temperatur [°C]\n")
    for t, temp in zip(times_min, temperatures):  # zip(): siehe Kapitel Datenstrukturen
        f.write(f"{t},{temp}\n")

print("Geschrieben:", protocol_file)

Geschrieben: data/messprotokoll.txt


Einige Details:

- `write()` schreibt einen String in die Datei – im Gegensatz zu `print()` **ohne** automatischen Zeilenumbruch. Zahlen wandelt man vorher in Strings um, am bequemsten mit f-Strings (z.B. `f"{temp:.1f}"`).
- `encoding="utf-8"` legt fest, wie Zeichen wie `°` oder `ä` gespeichert werden (siehe Kapitel *Datentypen*). Geben Sie es immer an, sonst verwendet Python das Standard-Encoding des Betriebssystems, und die Datei sieht auf einem anderen Rechner kaputt aus („Â°C“).

Mit dem Modus `"a"` hängen wir weitere Messwerte an, ohne die bisherigen zu verlieren – typisch für einen Datenlogger:

In [3]:
with open(protocol_file, "a", encoding="utf-8") as f:
    f.write("35,35.2\n")
    f.write("40,34.6\n")

## Textdateien lesen

Mit `f.read()` erhalten Sie den gesamten Inhalt einer geöffneten Datei als einen einzigen String.

Meist möchte man eine Datei aber **zeilenweise** verarbeiten. Dafür iteriert man einfach mit einer `for`-Schleife über das Dateiobjekt. Jede Zeile ist ein String, der noch das Zeilenende `\n` enthält; dieses entfernt man mit `strip()`. Anschließend zerlegt `split(",")` die Zeile in ihre Bestandteile – die aber immer noch Strings sind und erst in Zahlen umgewandelt werden müssen:

In [5]:
times = []
values = []

with open(protocol_file, encoding="utf-8") as f:  # "r" ist der Standard und darf entfallen
    for line in f:
        line = line.strip()
        if line == "" or line.startswith("#"):  # Leerzeilen und Kommentare überspringen
            continue
        t_str, temp_str = line.split(",")
        times.append(int(t_str))
        values.append(float(temp_str))

print("Zeiten:", times)
print("Temperaturen:", values)
print(f"Maximum: {max(values)} °C nach {times[values.index(max(values))]} min")

Zeiten: [0, 5, 10, 15, 20, 25, 30, 35, 40]
Temperaturen: [21.4, 23.9, 27.5, 31.2, 34.8, 36.1, 35.7, 35.2, 34.6]
Maximum: 36.1 °C nach 25 min


::: {.callout-tip}
### Lesen, verarbeiten, schreiben {.unnumbered}

Das Muster *Datei öffnen → zeilenweise lesen → Zeile bereinigen (`strip`) → zerlegen (`split`) → umwandeln (`int`, `float`) → in Listen speichern* begegnet Ihnen immer wieder.
:::

Für kleine Dateien bietet `Path` die Kurzformen `read_text()` und `write_text()`, die Öffnen und Schließen selbst erledigen.

<div class="exercises">
<b>Übungen:</b>

1. Schreiben Sie die Zahlen 1 bis 10 und ihre Quadrate in eine Datei `data/quadrate.txt`, eine Zeile pro Zahl im Format `3 -> 9`.
2. Lesen Sie die Datei wieder ein und berechnen Sie die Summe aller Quadratzahlen.

</div>

In [7]:
#| code-fold: true
#| output: false
# Übung 1
with open("data/quadrate.txt", "w", encoding="utf-8") as f:
    for n in range(1, 11):
        f.write(f"{n} -> {n**2}\n")

# Übung 2
total = 0
with open("data/quadrate.txt", encoding="utf-8") as f:
    for line in f:
        left, right = line.strip().split(" -> ")
        total += int(right)
print("Summe der Quadrate:", total)  # 385

Summe der Quadrate: 385


## CSV-Dateien mit dem Modul `csv`

CSV (*comma-separated values*) ist das wohl verbreitetste Austauschformat für Messdaten: eine Textdatei, in der jede Zeile einen Datensatz enthält und die Werte durch ein Trennzeichen getrennt sind. Die Prüfstandsdaten aus dem Kapitel *Bibliotheken* liegen in diesem Format vor:

In [8]:
csv_file = data_dir / "pruefstand.csv"

with open(csv_file, encoding="utf-8") as f:
    for i in range(3):
        print(f.readline().strip())  # readline(): genau eine Zeile lesen

zeit_s,drehzahl_rpm,drehmoment_Nm,temperatur_C,strom_A,spannung_V
0,1467.2924821489682,33.81031628726315,24.628518648371458,10.215086869820023,400.3974511569548
2,1461.4533926626523,35.36085692530741,25.32622786824618,10.628453711233902,400.8363769172621


Im Prinzip könnte man solche Dateien mit `split(",")` zerlegen. Das bricht aber, sobald ein Wert selbst ein Komma enthält (z.B. ein Kommentar in Anführungszeichen). Robuster ist das Modul `csv` aus der Standardbibliothek. Besonders praktisch ist der `DictReader`: Er liest die Kopfzeile und liefert jede weitere Zeile als Dictionary mit den Spaltennamen als Schlüsseln. CSV-Dateien öffnet man dabei immer mit `newline=""`, weil das Modul `csv` die Zeilenenden selbst behandelt.

In [9]:
import csv

max_temp = None
max_time = None
n_rows = 0

with open(csv_file, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        n_rows += 1
        temp = float(row["temperatur_C"])  # auch csv liefert nur Strings!
        if max_temp is None or temp > max_temp:
            max_temp = temp
            max_time = float(row["zeit_s"])

print(f"{n_rows} Datensätze gelesen")
print(f"Höchste Temperatur: {max_temp:.1f} °C bei t = {max_time:.0f} s")

500 Datensätze gelesen
Höchste Temperatur: 63.1 °C bei t = 988 s


Mit pandas genügt dafür eine Zeile (`df.loc[df["temperatur_C"].idxmax()]`); für tabellarische Messdaten ist pandas meist die bessere Wahl. Das Modul `csv` lohnt sich, wenn man ohne Zusatzpaket auskommen, Zeile für Zeile verarbeiten oder ein genau vorgegebenes Format schreiben muss.

::: {.callout-important}
### Typische Fehler beim Arbeiten mit Dateien {.unnumbered}

- **Falsches Arbeitsverzeichnis:** `FileNotFoundError`, obwohl die Datei „doch da ist“ – der relative Pfad bezieht sich auf einen anderen Ordner. Prüfen Sie mit `Path.cwd()`, wo Sie sich befinden.
- **Modus `"w"` statt `"a"`:** Die Datei wird ohne Rückfrage überschrieben.
- **Zahlen nicht umgewandelt:** Aus Dateien gelesene Werte sind immer Strings; `"21.4" + "1"` ergibt `"21.41"`.
- **CSV für das deutsche Excel:** Deutsches Excel erwartet Semikolon als Trennzeichen und Dezimalkomma, sonst landet alles in Spalte A. Abhilfe beim Schreiben: `csv.writer(f, delimiter=";")` zusammen mit `f"{x:.1f}".replace(".", ",")` und `encoding="utf-8-sig"`; beim Einlesen mit pandas `pd.read_csv(datei, sep=";", decimal=",")`.

Wie man auf Fehler wie eine fehlende Datei reagiert, statt dass das Programm abstürzt, zeigt Anhang B (Übung 3).
:::

<div class="exercises">
<b>Übung:</b>

3. Lesen Sie `data/pruefstand.csv` mit dem `csv.DictReader` ein und berechnen Sie für jeden Datensatz die mechanische Leistung $P = M \cdot 2\pi n / 60$ (in W, mit $M$ in Nm und $n$ in 1/min). Geben Sie die mittlere Leistung in kW aus.

</div>

In [ ]:
#| code-fold: true
#| output: false
import csv, math
from pathlib import Path

data_dir = Path("data")

# Übung 3
powers_W = []
with open(data_dir / "pruefstand.csv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        n = float(row["drehzahl_rpm"])
        M = float(row["drehmoment_Nm"])
        powers_W.append(M * 2 * math.pi * n / 60)
print(f"Mittlere mechanische Leistung: {sum(powers_W) / len(powers_W) / 1000:.2f} kW")

Mittlere mechanische Leistung: 5.36 kW
